# PhishForge - URL ML Model Training

Questo notebook addestra un modello di Machine Learning per identificare URL phishing.

In [3]:
# Step 1: Installa le librerie necessarie (se non già installate)
import sys
!{sys.executable} -m pip install scikit-learn joblib pandas numpy -q

print("✅ Librerie installate")

✅ Librerie installate


In [4]:
# Step 2: Crea dataset di training
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import joblib
from pathlib import Path

# Dataset di esempio con URL phishing e legittimi
phishing_urls = [
    "http://paypal-verify.com/secure",
    "https://amazon-login.net/account",
    "http://bit.ly/free-iphone",
    "https://secure-banking.xyz/login",
    "http://facebook-security.com/verify",
    "https://apple-id.support.com/unlock",
    "http://netflix-payment.net/update",
    "https://paypal-secure.xyz/confirm",
    "http://google-security.com/alert",
    "https://microsoft-account.net/verify",
    "http://account-verification.com/paypal",
    "https://secure-login.net/banking",
    "http://win-prize.com/claim",
    "https://urgent-action.com/account",
    "http://verify-identity.net/secure"
]

legitimate_urls = [
    "https://www.paypal.com/signin",
    "https://www.amazon.com/account",
    "https://www.apple.com/iphone",
    "https://www.chase.com/personal/online-banking",
    "https://www.facebook.com/login",
    "https://appleid.apple.com",
    "https://www.netflix.com/billing",
    "https://www.paypal.com",
    "https://www.google.com",
    "https://www.microsoft.com/account",
    "https://github.com/login",
    "https://stackoverflow.com",
    "https://www.wikipedia.org",
    "https://www.reddit.com",
    "https://www.linkedin.com"
]

# Crea DataFrame
df = pd.DataFrame({
    'url': phishing_urls + legitimate_urls,
    'is_phishing': [1] * len(phishing_urls) + [0] * len(legitimate_urls)
})

print(f"✅ Dataset creato: {len(df)} URL totali")
print(f"   - Phishing: {sum(df['is_phishing'])} URL")
print(f"   - Legittimi: {len(df) - sum(df['is_phishing'])} URL")
print("\nEsempi:")
print(df.head())

✅ Dataset creato: 30 URL totali
   - Phishing: 15 URL
   - Legittimi: 15 URL

Esempi:
                                   url  is_phishing
0      http://paypal-verify.com/secure            1
1     https://amazon-login.net/account            1
2            http://bit.ly/free-iphone            1
3     https://secure-banking.xyz/login            1
4  http://facebook-security.com/verify            1


In [5]:
# Step 3: Addestra il modello
X_train, X_test, y_train, y_test = train_test_split(
    df['url'], df['is_phishing'], test_size=0.2, random_state=42
)

# Crea e addestra il vectorizer
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Addestra il modello
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

# Valuta il modello
train_score = model.score(X_train_vec, y_train)
test_score = model.score(X_test_vec, y_test)

print(f"✅ Modello addestrato!")
print(f"   - Training accuracy: {train_score:.2%}")
print(f"   - Test accuracy: {test_score:.2%}")

✅ Modello addestrato!
   - Training accuracy: 100.00%
   - Test accuracy: 100.00%


In [6]:
# Step 4: Salva il modello e il vectorizer
MODEL_PATH = Path.cwd() / "ml_url_model.joblib"
VECTORIZER_PATH = Path.cwd() / "ml_url_vectorizer.joblib"

joblib.dump(model, MODEL_PATH)
joblib.dump(vectorizer, VECTORIZER_PATH)

print(f"✅ Modello salvato in: {MODEL_PATH}")
print(f"✅ Vectorizer salvato in: {VECTORIZER_PATH}")
print(f"\n📊 Dimensione file:")
print(f"   - Modello: {MODEL_PATH.stat().st_size / 1024:.2f} KB")
print(f"   - Vectorizer: {VECTORIZER_PATH.stat().st_size / 1024:.2f} KB")

✅ Modello salvato in: /workspaces/PhishForge-Lite/PhishForge/ml/ml_url_model.joblib
✅ Vectorizer salvato in: /workspaces/PhishForge-Lite/PhishForge/ml/ml_url_vectorizer.joblib

📊 Dimensione file:
   - Modello: 92.51 KB
   - Vectorizer: 17.20 KB


In [7]:
import joblib
from pathlib import Path
import logging

# In un notebook, usa il path corrente invece di __file__
BASE_DIR = Path.cwd()
MODEL_PATH = BASE_DIR / "ml_url_model.joblib"
VECTORIZER_PATH = BASE_DIR / "ml_url_vectorizer.joblib"

print(f"BASE_DIR: {BASE_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"VECTORIZER_PATH: {VECTORIZER_PATH}")

BASE_DIR: /workspaces/PhishForge-Lite/PhishForge/ml
MODEL_PATH: /workspaces/PhishForge-Lite/PhishForge/ml/ml_url_model.joblib
VECTORIZER_PATH: /workspaces/PhishForge-Lite/PhishForge/ml/ml_url_vectorizer.joblib


In [8]:
# Carico una volta sola, all'avvio
try:
    ML_MODEL = joblib.load(MODEL_PATH)
    ML_VECTORIZER = joblib.load(VECTORIZER_PATH)
    print("✅ ML model and vectorizer loaded correctly.")
except Exception as e:
    print(f"❌ Error loading ML artifacts: {e}")
    ML_MODEL = None
    ML_VECTORIZER = None


def ml_score_url(url: str) -> float:
    """Restituisce uno score 0–100 basato sul modello ML."""
    try:
        if ML_MODEL is None or ML_VECTORIZER is None:
            return 0.0

        X = ML_VECTORIZER.transform([url])
        proba = ML_MODEL.predict_proba(X)[0][1]  # probabilità di phishing (classe 1)

        return round(proba * 100, 2)
    except Exception as e:
        print(f"Error computing ML score for URL '{url}': {e}")
        return 0.0

print("✅ Function ml_score_url defined")

✅ ML model and vectorizer loaded correctly.
✅ Function ml_score_url defined


In [9]:
# Test con alcuni URL
test_urls = [
    "https://paypal-secure.xyz/verify",
    "https://www.google.com",
    "http://bit.ly/free-money",
    "https://amazon.com"
]

print("🧪 Testing ML URL scoring:\n")
for url in test_urls:
    score = ml_score_url(url)
    print(f"URL: {url:45} → Score: {score:.2f}/100")

🧪 Testing ML URL scoring:

URL: https://paypal-secure.xyz/verify              → Score: 99.00/100
URL: https://www.google.com                        → Score: 4.00/100
URL: http://bit.ly/free-money                      → Score: 81.00/100
URL: https://amazon.com                            → Score: 30.00/100
